# 02. 상담 예제 1 — "카드가 안 긁히는데 어떻게 해야 하나요?" (clarification loop)

첫 번째 상담 예제입니다. 고객 문의에 **구체 정보가 없어서**, agent가 D5(정보 충분성)에서 `need_clarification`을 판정하고 **되묻는 질문을 생성 -> 고객 답변 수신 -> 재판정**하는 recursive loop를 돕니다.

예상 경로:
```
D1(guardrail) -> D2(routing) -> D5(need_clarification) -> 되묻기/답변 -> D5(info_sufficient)
-> D3(tool 선택) -> tool 성공 -> D6(goal_met) -> 요약 -> D7(no_risk) -> D4(resolve) -> 최종 응답
```

같은 질문을 **① Haiku 판단**으로, **② strands-decider 판단**으로 각각 실행하며 STEP별 출력을 관찰하고, 마지막에 수행시간을 비교합니다. (응답 생성은 양쪽 모두 Sonnet 4.5 — 차이는 판단 계층뿐)

In [ ]:
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common 경로 (repo 내 상대 위치)
import ws_common as W
graphs, client = W.make_graphs()
sc = W.SCENARIOS[1]
print("시나리오:", sc["title"])
print("고객 질문:", sc["question"])
print("script(실습 재현용 사전 설정):", sc["script"])

## 전체 workflow에서 이 시나리오의 위치

이번 시나리오가 지나는 경로(빨간 테두리): 접수 심사 -> 업무 분류 -> **되묻기 loop(clarify_gate <-> ask_clarify)** -> tool 1회 성공 -> 요약 -> 리스크 이상 없음 -> 종결. KYC sub-flow와 give_up 경로는 이번에는 지나가지 않습니다.

배정 상담팀(빨간 표시): **cards(카드 발급/배송)** — 카드 오작동 문의이므로 cards 팀 tool(card_troubleshoot)로 처리됩니다.
> 참고: 이 다이어그램은 실행되는 LangGraph에서 자동 생성된 것이 아니라, `ws_common.py`의 한글 캡션 사전(`KO_NODES`/`KO_EDGES`)을 노드 이름 기준으로 그린 **시각화 전용 그림**입니다 (상세 설명: `[KR]_01_build_agent.ipynb` 참조).


In [ ]:
W.draw_graph(highlight=W.SCENARIO_PATHS[1], show_teams=True, width=760)

## ① Haiku 4.5가 판단하는 agent로 실행

각 STEP에서 어떤 노드가 어떤 판정(label)을 몇 ms에 내리는지, 되묻는 질문과 고객 답변, 최종 응답까지 그대로 출력됩니다. **판정 한 번마다 600ms 이상** 걸리는 것을 눈여겨보세요.

In [ ]:
res_haiku = W.run_verbose(graphs["Haiku"], sc["question"], sc["script"], "Haiku")

## ② strands-decider(E4B) 판단으로 실행

같은 그래프, 같은 질문입니다. 판정이 LLM 왕복 없이 로컬 서버의 forward pass 1회로 끝나고 **confidence**가 함께 찍히는 점이 다릅니다.

In [ ]:
res_strands = W.run_verbose(graphs["StrandsDecider"], sc["question"], sc["script"], "StrandsDecider")

## ③ 수행시간 비교

생성(Sonnet) 시간은 양쪽이 비슷하고, **판단 계층의 시간이 통째로 사라지는 것**이 핵심입니다.

In [ ]:
W.compare(res_haiku, res_strands)

**관찰 포인트**
- 되묻기 loop 때문에 D5가 두 번 실행됩니다 — loop가 돌수록 판단 횟수가 늘고, 판단이 느린 backend일수록 누적 격차가 커집니다.
- 두 arm의 판정 label이 같은지 비교해 보세요(비교표 우측). label이 같다면 이 대화에서는 품질 손실 없이 속도만 얻은 것입니다.
- 다음: `[KR]_03_scenario2_retry.ipynb` — tool이 실패했을 때 agent가 스스로 복구하는 loop.